In [18]:
# =====================================================
# 연습문제 5 (응용/해양학): 해수 밀도 계산기
# -----------------------------------------------------
# 목표: 해양학에서 가장 기본이 되는 해수 밀도 ρ(S, T, 0)
#       (UNESCO 1980 / EOS-80, 압력 = 0 dbar 표면식)을
#       클래스로 캡슐화하고 GUI 에서 사용한다.
#
# 입력:
#   T : 수온 [°C]        (보통 -2 ~ 35)
#   S : 염분 [PSU]       (보통 0 ~ 42)
# 출력:
#   ρ      [kg/m³]
#   σ_t    [kg/m³]  =  ρ - 1000
#
# 공식 (UNESCO 1980, 압력 0):
#   ρ_w(T) = 999.842594 + 6.793952e-2*T - 9.095290e-3*T^2
#            + 1.001685e-4*T^3 - 1.120083e-6*T^4 + 6.536336e-9*T^5
#   A = 8.24493e-1 - 4.0899e-3*T + 7.6438e-5*T^2 - 8.2467e-7*T^3 + 5.3875e-9*T^4
#   B = -5.72466e-3 + 1.0227e-4*T - 1.6546e-6*T^2
#   C = 4.8314e-4
#   ρ(S, T) = ρ_w(T) + A*S + B*S^1.5 + C*S^2
#
# 주의(오류 가능 지점):
#   - 거듭제곱 ** 와 곱셈 * 를 헷갈리지 않도록 주의 (포트란 ** 와 동일)
#   - Entry 에서 받은 문자열은 float() 로 변환 ("23.5" → 23.5). 실패 시 ValueError.
#   - S^1.5 은 S**1.5 이며, S 가 음수면 복소수가 된다.
#     → 유효 범위 (T: -2~40, S: 0~42) 를 클래스에서 검증.
# =====================================================

from tkinter import *
from tkinter import messagebox



class SeaWater:
    """UNESCO 1980 EOS-80 (압력 0 dbar) 기반 해수 밀도 계산기."""

    T_MIN, T_MAX = -2.0, 40.0    # ← 이 줄이 빠져 있었음
    S_MIN, S_MAX =  0.0, 42.0    # ← 이 줄도 빠져 있었음

    def __init__(self, T, S):
        if not (self.T_MIN <= T <= self.T_MAX):   # T_Min → T_MIN
            raise ValueError(f"T는 {self.T_MIN}~{self.T_MAX}°C 범위여야 합니다.")
        if not (self.S_MIN <= S <= self.S_MAX):    # s → S
            raise ValueError(f"S는 {self.S_MIN}~{self.S_MAX} PSU 범위여야 합니다.")
        self.T = T
        self.S = S
  

    def density(self):
        T, S = self.T, self.S

        # 순수 물 밀도 (S = 0)
        rho_w = (999.842594 + 6.793952e-2 * T - 9.095290e-3 * T**2 + 1.001685e-4 * T**3 - 1.120083e-6 * T**4 + 6.536336e-9 * T**5)

        A = (8.24493e-1 - 4.0899e-3 * T + 7.6438e-5 * T**2 - 8.2467e-7 * T**3 + 5.3875e-9 * T**4)

        B = (-5.72466e-3 + 1.0227e-4 * T - 1.6546e-6 * T**2)

        C = 4.8314e-4

        rho = rho_w + A * S + B * S**1.5 + C * S * S
        return rho


    def sigma_t(self):
        """σ_t = ρ - 1000  [kg/m^3]"""
        return self.density() - 1000.0

    def __repr__(self):
        # 강의 12장 __repr__() 와 동일한 패턴
        return f"SeaWater(T={self.T:.2f}°C, S={self.S:.2f}PSU)"


## 함수 선언 부분 ##
def on_calc():
    try:
        T = float(entry_T.get())
        S = float(entry_S.get())
    except ValueError:
        messagebox.showerror("입력 오류", "T와 S는 숫자로 입력해 주세요.")
        return

    try:
        sw = SeaWater(T, S)
    except ValueError as e:
        messagebox.showerror("범위 오류", str(e))
        return

    rho = sw.density()
    sigma = sw.sigma_t()
    result.config(text=f"ρ   = {rho:9.4f} kg/m³\nσ_t = {sigma:9.4f} kg/m³")


## 메인 코드 부분 ##
window = Tk()
window.title("연습 5 - 해수 밀도 계산기 (EOS-80)")
window.geometry("380x230")

Label(window, text="수온 T [°C]:").grid(row=0, column=0, padx=10, pady=8, sticky="e")
entry_T = Entry(window, width=12);  entry_T.grid(row=0, column=1, padx=10, pady=8)
entry_T.insert(0, "15.0")

Label(window, text="염분 S [PSU]:").grid(row=1, column=0, padx=10, pady=8, sticky="e")
entry_S = Entry(window, width=12);  entry_S.grid(row=1, column=1, padx=10, pady=8)
entry_S.insert(0, "34.5")

Button(window, text="계산", width=12, command=on_calc).grid(row=2, column=0, columnspan=2, pady=10)

result = Label(window, text="", font=("Consolas", 12), justify=LEFT)
result.grid(row=3, column=0, columnspan=2)

window.mainloop()

In [22]:
# =====================================================
# 연습문제 6 (응용/해양학): CTD 데이터 뷰어
# -----------------------------------------------------
# 목표:
#   1) CSV 파일에서 (수심, 수온, 염분) 데이터를 읽어 CTD 클래스 객체로 보관
#   2) tkinter Menu "파일 > 열기 / 종료" 로 데이터 파일을 선택
#   3) Canvas 에 수온/염분의 수직 프로파일을 두 색으로 그린다
#   4) Radiobutton 으로 어떤 변수를 그릴지 전환
#
# 입력 CSV 형식 (header 1줄, 단위 포함):
#   depth_m,temperature_C,salinity_PSU
#   0,22.50,33.80
#   ...
#
# 주의(오류 가능 지점):
#   * Canvas 좌표는 y 가 아래로 증가 → 수심도 아래로 증가하므로 그대로 그려도 자연스럽다.
#     단, 수온/염분 축은 좌표 변환을 잘 해 주어야 한다.
#   * Entry 가 아니라 파일에서 읽을 때도 변환 실패(ValueError)는 항상 가능 → try/except.
#   * 윈도 크기가 바뀌어도 다시 그릴 수 있도록 draw_profile() 을 함수로 분리.
#   * 사용 예: 이 파일과 같은 폴더의 sample_ctd.csv 를 열어 보세요.
# =====================================================

from tkinter import *
from tkinter import filedialog, messagebox
import csv


## 클래스 선언 부분 ##
class CTD:
    """CTD 캐스트 한 회 분 데이터 (수심, 수온, 염분)"""

    def __init__(self, depth=None, temperature=None, salinity=None, name=""):
        self.depth = depth or []        # m
        self.temperature = temperature or []   # °C
        self.salinity = salinity or []  # PSU
        self.name = name

    @classmethod
    def from_csv(cls, path):
        """CSV 파일에서 CTD 객체 생성 (분류 메서드 = classmethod 사용 예)"""
        depth, temp, sal = [], [], []
        with open(path, "r", encoding="utf-8") as f:
            reader = csv.reader(f)
            header = next(reader)   # header 행 건너뛰기
            for row in reader:
                if not row:
                    continue
                depth.append(float(row[0]))
                temp.append(float(row[1]))
                sal.append(float(row[2]))
        return cls(depth, temp, sal, name=path.split("/")[-1])
                

    def avg_T(self):
        return sum(self.temperature) / len(self.temperature) if self.temperature else 0.0
        
    def avg_S(self):
        return sum(self.salinity) / len(self.salinity) if self.salinity else 0.0
        

    def max_depth(self):
        return max(self.depth) if self.depth else 0.0
      
    def __repr__(self):
        return f"CTD(n={len(self.depth)}, name={self.name!r})"    


## 함수 선언 부분 ##
ctd = None     # 현재 표시 중인 CTD 객체


def open_file():
    global ctd
    path = filedialog.askopenfilename(
        parent=window,
        title="CTD CSV 파일 열기",
        filetypes=(("CSV 파일", "*.csv"), ("모든 파일", "*.*"))
    )
    if not path:
        return
    try:
        ctd = CTD.from_csv(path)
    except Exception as e:
        messagebox.showerror("파일 오류", f"파일을 읽지 못했습니다.\n{e}")
        return

    info.config(
        text=f"파일: {ctd.name}\n"
             f"포인트 수: {len(ctd.depth)}    최대 수심: {ctd.max_depth():.1f} m\n"
             f"평균 수온: {ctd.avg_T():.2f} °C    평균 염분: {ctd.avg_S():.2f} PSU"
    )
    draw_profile()


def exit_app():
    window.quit()
    window.destroy()


def draw_profile(*_):
    canvas.delete("all")
    if ctd is None or not ctd.depth:
        return

    var = var_choice.get()    # 1: 수온, 2: 염분
    values = ctd.temperature if var == 1 else ctd.salinity
    color  = "red" if var == 1 else "blue"
    label_text = "수온 [°C]" if var == 1 else "염분 [PSU]"

    W = canvas.winfo_width()
    H = canvas.winfo_height()
    if W < 50 or H < 50:
        return

    pad = 50
    x_min, x_max = min(values) - 0.5, max(values) + 0.5
    d_min, d_max = 0, ctd.max_depth()

    def to_px(v, d):
        x = pad + (v - x_min) / (x_max - x_min) * (W - 2 * pad)
        y = pad + (d - d_min) / (d_max - d_min) * (H - 2 * pad)
        return x, y

    # 축
    canvas.create_rectangle(pad, pad, W - pad, H - pad, outline="black")
    canvas.create_text(W / 2, 20, text=label_text, font=("맑은 고딕", 11, "bold"))
    canvas.create_text(20, H / 2, text="수심 [m]", angle=90, font=("맑은 고딕", 11, "bold"))

    # 데이터 라인
    pts = []
    for v, d in zip(values, ctd.depth):
        pts.append(to_px(v, d))
    for (x1, y1), (x2, y2) in zip(pts[:-1], pts[1:]):
        canvas.create_line(x1, y1, x2, y2, fill=color, width=2)
    for x, y in pts:
        canvas.create_oval(x - 3, y - 3, x + 3, y + 3, fill=color, outline="")


## 메인 코드 부분 ##
window = Tk()
window.title("연습 6 - CTD Profile Viewer")
window.geometry("600x520")

# 메뉴
menubar = Menu(window)
window.config(menu=menubar)
filemenu = Menu(menubar, tearoff=0)
menubar.add_cascade(label="파일", menu=filemenu)
filemenu.add_command(label="열기 ...", command=open_file)
filemenu.add_separator()
filemenu.add_command(label="종료", command=exit_app)

# 정보 영역
info = Label(window, text="파일을 열어 주세요.", justify=LEFT, anchor="w",
             font=("맑은 고딕", 10))
info.pack(fill=X, padx=10, pady=5)

# 변수 선택
var_choice = IntVar(value=1)
frm = Frame(window)
frm.pack(anchor="w", padx=10)
Radiobutton(frm, text="수온",  variable=var_choice, value=1, command=draw_profile).pack(side=LEFT)
Radiobutton(frm, text="염분",  variable=var_choice, value=2, command=draw_profile).pack(side=LEFT)

# 그래프 영역
canvas = Canvas(window, bg="white")
canvas.pack(fill=BOTH, expand=True, padx=10, pady=10)
canvas.bind("<Configure>", draw_profile)   # 창 크기가 바뀌면 다시 그림

window.mainloop()

In [26]:
# =====================================================
# 연습문제 7 (응용/해양학): 조석 시뮬레이터
# -----------------------------------------------------
# 목표:
#   1) 조석을 조화분해 한 결과(주요 4개 분조)를 클래스로 모형화
#        Constituent(name, amplitude, period_hr, phase_deg)
#   2) Tide 클래스는 여러 Constituent 를 합쳐 시계열 η(t) 를 만든다.
#   3) Canvas 에 시간에 따른 조위 곡선을 애니메이션으로 그린다.
#        (after() 사용. threading 으로 GUI 갱신은 금지!)
#   4) Checkbutton 으로 각 분조의 켜기/끄기 가능
#
# 주요 분조 표준 주기 (시간):
#   M2  : 12.4206    (반일주 주조석)
#   S2  : 12.0000    (반일주, 태양)
#   K1  : 23.9345    (일주, 달+태양)
#   O1  : 25.8193    (일주, 달)
#
# 식:
#   η(t) = Σ_i  A_i * cos( 2π * t / T_i  -  φ_i )
#   (t: 시간 [hr],  A: 진폭 [m],  T: 주기 [hr],  φ: 위상 [rad])
#
# 주의(오류 가능 지점):
#   * Python math.cos 는 라디안 입력 → 위상을 도(deg) 로 주면 math.radians() 변환 필요.
#   * after(ms, fn) 의 ms 는 정수 (밀리초). 너무 작으면 GUI 가 끊기지 않고 부드럽지만 CPU 사용 증가.
#   * 같은 변수에 새 PhotoImage 등을 매번 할당하면 가비지 컬렉터가 가져가 그림이 안 보일 수 있다.
#     (본 문제에선 직접적이지 않지만 일반 GUI 작성 시 유의)
# =====================================================

from tkinter import *
import math


## 클래스 선언 부분 ##
class Constituent:
    """단일 조화 분조"""
    def __init__(self, name, amplitude, period_hr, phase_deg=0.0):
        self.name = name
        self.A = amplitude
        self.T = period_hr
        self.phi = math.radians(phase_deg)
        self.enabled = True

    def eta(self, t):
        if not self.enabled:
            return 0.0
        return self.A*math.cos(2*math.pi * t / self.T - self.phi)
    def __repr__(self):
        return f"{self.name}(A={self.A}, T={self.T})"


class Tide:
    """여러 분조의 합으로 조위를 계산"""

    def __init__(self, constituents):
        self.constituents = constituents

    def eta(self, t):
        return sum(c.eta(t) for c in self.constituents)
    

    def by_name(self, name):
        for c in self.constituents:
            if c.name == name:
                return c
        return None


## 함수 선언 부분 ##
def toggle(name):
    """체크박스로 분조 on/off"""
    c = tide.by_name(name)
    if c is None:
        return
    c.enabled = bool(check_vars[name].get())


def step():
    """다음 프레임을 그린다 (애니메이션)"""
    global current_t
    current_t += 0.25   # 0.25 시간씩 진행
    draw()
    window.after(50, step)   # 50 ms 마다 갱신


def draw():
    canvas.delete("all")
    W = canvas.winfo_width()
    H = canvas.winfo_height()
    if W < 50 or H < 50:
        return

    # 표시할 시간 창: [current_t - 24h, current_t + 12h]
    t_left  = current_t - 24
    t_right = current_t + 12
    t_span  = t_right - t_left

    pad = 40
    y_mid = H / 2

    # 진폭 자동 결정 (현재 enabled 인 분조의 진폭 합)
    A_total = sum(c.A for c in tide.constituents if c.enabled) + 0.1
    y_scale = (H / 2 - pad) / A_total

    def to_px(t, eta):
        x = pad + (t - t_left) / t_span * (W - 2 * pad)
        y = y_mid - eta * y_scale
        return x, y

    # 가로축 (해수면 0 m 기준선)
    canvas.create_line(pad, y_mid, W - pad, y_mid, fill="#888888", dash=(2, 2))

    # 곡선
    pts = []
    n = 300
    for i in range(n + 1):
        t = t_left + t_span * i / n
        pts.append(to_px(t, tide.eta(t)))
    for (x1, y1), (x2, y2) in zip(pts[:-1], pts[1:]):
        canvas.create_line(x1, y1, x2, y2, fill="navy", width=2)

    # 현재 시각 (수직선)
    x_now, _ = to_px(current_t, 0)
    canvas.create_line(x_now, pad, x_now, H - pad, fill="red", width=1, dash=(4, 2))
    eta_now = tide.eta(current_t)
    canvas.create_text(x_now + 10, pad + 10,
                       text=f"t = {current_t:6.2f} h\nη = {eta_now:+.3f} m",
                       anchor="w", font=("Consolas", 10))


## 메인 코드 부분 ##
tide = Tide([
    Constituent("M2", 0.80, 12.4206, 0),
    Constituent("S2", 0.30, 12.0000, 30),
    Constituent("K1", 0.25, 23.9345, 60),
    Constituent("O1", 0.18, 25.8193, 90),
])

current_t = 0.0

window = Tk()
window.title("연습 7 - 조석 시뮬레이터")
window.geometry("640x460")

# 분조 on/off 체크박스
top = Frame(window)
top.pack(fill=X)
check_vars = {}
for c in tide.constituents:
    v = IntVar(value=1)
    check_vars[c.name] = v
    Checkbutton(top, text=f"{c.name}  A={c.A:.2f}m T={c.T:.2f}h",
                variable=v, command=lambda n=c.name: toggle(n)).pack(side=LEFT, padx=5)

# 그래프
canvas = Canvas(window, bg="white")
canvas.pack(fill=BOTH, expand=True, padx=10, pady=10)

step()        # 애니메이션 시작
window.mainloop()